# XGBoosted + Prophet Hybrid Model

We saw in 1modeling_experiments.ipynb that XGBoost+Prophet performed well. The margins between Prophet and the XGBoosted Prophet models were quite slim. We do some feature engineering and hyperparameter tuning to see if we can improve performance.

## Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import xgboost as xgb
from xgboost import plot_importance

In [2]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

rs = rs[rs['borough']=='MANHATTAN']

In [3]:
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date'] < '2025-03-01']
rs = rs[rs['created_date'] >= '2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

# fill missing dates with 0
full_dates = pd.date_range(start='2020-01-01', end='2025-02-28', freq='D').date
rs = rs.set_index('ds').reindex(full_dates, fill_value=0).rename_axis('ds').reset_index()
rs

,ds,y
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
1881,2025-02-24,19
1882,2025-02-25,20
1883,2025-02-26,15
1884,2025-02-27,16


**Warning:** One should be cautious about temporal leakage. Our goal is to forecast 14 days out into the future. So, we should not use something like a lagged feature by 7 days. The reason is that when we are forecasting the 8th day out and onwards, we would be using a feature that is still outside of our scope. In other words, we should really avoid any lagged features of <14 days for our purposes.

## First Run

In [ ]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    # Must shift by 14 days because we do not want to let there be temporal leakage in our evaluations
    df['ma7'] = df['y'].shift(14).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(14).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(14).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(14).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(14).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(14).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(14).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(14).rolling(window=365).mean()
    
    return df


In [ ]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [ ]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Remove overlapping columns to avoid join errors
    overlap = wd.columns.intersection(df.columns)
    wd = wd.drop(columns=overlap)
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [ ]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [ ]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [ ]:
def add_new_lags(df, x):
    # lags
    target_map = df[x].to_dict()
    df[f'{x}lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag17'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag18'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag19'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag20'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag21'] = (df.index - pd.Timedelta('16 days')).map(target_map)

    df[f'{x}lag30'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag365'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag730'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    return df

In [ ]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'dayofyear', 'temperature_2m_max_lag14', 'temperature_2m_max_lag30',
            'temperature_2m_max_lag60', 
            'is_federal_holiday', 
            'lag15', 'lag16', 'lag30', 'lag60', 'lag90', 'lag120', 'lag150', 
            'lag180', 
            'lag362', 'lag363', 'lag364', 'lag365', 'lag366', 'lag367',
            'residualslag15', 'residualslag16', 'residualslag17',
            'residualslag18', 'residualslag19', 'residualslag20', 'residualslag21',
            'residualslag30', 'residualslag365', 'residualslag730', 
            'trend', 'yhat_lower', 'yhat_upper', 
            ]

In [ ]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': 0.5, 
         'n_estimators': 200, 
        #  'min_child_weight': 6, 
         'learning_rate': 0.01,
        # 'max_depth': 6, 
        # 'subsample': 1,
        # 'colsample_bytree': 0.96,
        # 'colsample_bylevel': 0.6, 
        # 'colsample_bynode': 0.9, 
        # 'reg_alpha': 2.2, 
        # 'gamma': 100, 
        # 'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }

In [ ]:
save = rs['ds'].copy().values
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)
rs['ds']=save
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')
rs.columns

In [ ]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset into training and testing sets
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet on the training data
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Make predictions on the training set to calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
    train_forecast = model.predict(train_future)
    
    # Calculate residuals (actual - predicted) on the training data
    train_residuals = train['y'].values - train_forecast['yhat'].values
    
#    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    # Build a new DataFrame of residuals
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

    train.loc[:, 'residuals'] = train_residuals
    add_new_lags(train, 'residuals')

    train.loc[:, 'trend'] = train_forecast['trend'].values
    train.loc[:, 'yhat_lower'] = train_forecast['yhat_lower'].values
    train.loc[:, 'yhat_upper'] = train_forecast['yhat_upper'].values
    
    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_residuals, y_train_residuals)

    test.loc[:, 'residuals'] = np.nan    
    dummy = pd.concat([X_train_residuals, test], axis=0)  # row-wise   
    add_new_lags(dummy,'residuals')

    test = dummy.iloc[test_index]

    # Forecast using Prophet on the test set
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)

    
    # Predict residuals using XGBoost for the test set
    test.loc[:, 'trend'] = prophet_forecast[-len(test):]['trend'].values
    test.loc[:, 'yhat_lower'] = prophet_forecast[-len(test):]['yhat_lower'].values
    test.loc[:, 'yhat_upper'] = prophet_forecast[-len(test):]['yhat_upper'].values
    
    X_test = test[FEATURES]  # Features for the test set
    xgb_residual_preds = xgb_model.predict(X_test)
    
    
    
    # Combine Prophet's forecast and XGBoost's residual prediction
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Store the results for this fold
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
    # # Uncomment code below if you want to have plots on feature importance. I'll leave it commented out for obvious reasons.
    # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
    # plot_importance(xgb_model, ax=ax1, importance_type='gain')
    # ax1.set_title('Gain-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax2, importance_type='weight')
    # ax2.set_title('Split-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax3, importance_type='cover')
    # ax3.set_title('Cover Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
    # ax4.set_title('Total Gain Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
    # ax5.set_title('Total Cover Importance', fontsize=12)

    plt.show()

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

In [ ]:
train.columns

In [ ]:
prophet_xgb_results_df

# Examining Residuals Data

In [ ]:
model = Prophet(holidays=holidays)
model.add_country_holidays(country_name='US')
model.fit(rs)
rs_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
rs_forecast = model.predict(rs_future)
    
rs_residuals = rs['y'].values - rs_forecast['yhat'][:len(rs)].values.astype(int)
    

In [ ]:
residuals = pd.DataFrame({'ds': rs['ds'],  # Use the 'ds' column from the original `rs` dataframe
    'y': rs_residuals  # Use the computed residuals
})
residuals['y'].describe()

In [ ]:
plt.figure(figsize=(50, 12))
plt.plot(residuals_df['ds'], residuals_df['y'], marker='o', linestyle='', color='b', label='Residuals', markersize = 10)
plt.title('Residuals vs Date')
plt.xlabel('Date', size =20)
plt.ylabel('Residuals', size =30)
plt.grid(True)
plt.xticks(rotation=45, size= 20)  # Rotate x-axis labels for better readability
plt.yticks(size=20)
plt.legend()
plt.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

window_size = 30

residuals_df['rolling_mean'] = residuals_df['y'].rolling(window=window_size).mean()
residuals_df['rolling_variance'] = residuals_df['y'].rolling(window=window_size).var()
residuals_df['y_365_days_ago'] = residuals_df['y'].shift(365)
residuals_df['y_15_days_ago'] = residuals_df['y'].shift(15)



fig = go.Figure()

fig.add_trace(go.Scatter(
    x=residuals_df['ds'],
    y=residuals_df['y'],
    mode='markers',
    name='Residuals',
    marker=dict(size=5, color='blue') 
))

# fig.add_trace(go.Scatter(
#     x=residuals_df['ds'],
#     y=residuals_df['rolling_mean'],
#     mode='lines',
#     name=f'Rolling Mean (window={window_size})',
#     line=dict(color='red', width=3)
# ))

# fig.add_trace(go.Scatter(
#     x=residuals_df['ds'],
#     y=residuals_df['rolling_variance'],
#     mode='lines',
#     name=f'Rolling Variance (window={window_size})',
#     line=dict(color='green', width=3)
# ))

# Plot 365 days ago actuals as orange dots
fig.add_trace(go.Scatter(
    x=residuals_df['ds'],
    y=residuals_df['y_365_days_ago'],
    mode='markers',  # Change mode to 'markers' to plot dots
    name='365 Days Ago Actuals',
    marker=dict(color='orange', size=4)  # Customize the dot color and size
))

# fig.add_trace(go.Scatter(
#     x=residuals_df['ds'],
#     y=residuals_df['y_15_days_ago'],
#     mode='markers',  # Change mode to 'markers' to plot dots
#     name='15 Days Ago Actuals',
#     marker=dict(color='green', size=4)  # Customize the dot color and size
# ))


# Update layout
fig.update_layout(
    title='Residuals and 365 Days Ago Actuals vs Date',
    xaxis_title='Date',
    yaxis_title='Residuals',
    xaxis=dict(tickangle=45, tickfont=dict(size=20)),
    yaxis=dict(tickfont=dict(size=20)),
    legend=dict(font=dict(size=10),  # Further reduced legend size
                x=0.8, y=0.95,  # Legend position
                traceorder='normal',
                bgcolor='rgba(255, 255, 255, 0.7)',  # Background for legend
                bordercolor='Black', borderwidth=1),
    width=1000,  # Width of the plot
    height=600,  # Height of the plot
    margin=dict(l=100, r=100, t=100, b=100), 
    template='plotly'
)

# Show the figure
fig.show()

# window_size = 730
# residuals_df['rolling_mean'] = residuals_df['y'].rolling(window=window_size).mean()
# residuals_df['rolling_variance'] = residuals_df['y'].rolling(window=window_size).var()

# plt.figure(figsize=(60, 12))

# plt.plot(residuals_df['ds'], residuals_df['y'], marker='o', linestyle='', color='b', label='Residuals', markersize=10)
# plt.plot(residuals_df['ds'], residuals_df['rolling_mean'], color='r', label=f'Rolling Mean (window={window_size})', linewidth=3)
# plt.plot(residuals_df['ds'], residuals_df['rolling_variance'], color='g', label=f'Rolling Variance (window={window_size})', linewidth=3)

# # Add title and labels
# plt.title('Residuals, Rolling Mean, and Rolling Variance vs Date')
# plt.xlabel('Date', size=20)
# plt.ylabel('Residuals', size=30)
# plt.grid(True)
# plt.xticks(rotation=45, size=20)  # Rotate x-axis labels for better readability
# plt.yticks(size=20)

# # Show legend
# plt.legend()

# # Show plot
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm


sm.graphics.tsa.plot_acf(residuals_df['y'], lags=24)
sm.graphics.tsa.plot_pacf(residuals_df['y'], lags=24)
plt.show()

# Optuna Hyperparameter Tuning


In the XGBoost + Prophet hybrid model, we can tune a lot of parameters and also do a lot of feature engineering. For this reasoning, we shall use Optuna. Optuna also has the added benefit that it can record its studies in a .db so that future studies can piggyback off of old studies.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import xgboost as xgb
from xgboost import plot_importance
import optuna
from optuna.exceptions import TrialPruned

In [5]:
# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs[rs['borough']=='MANHATTAN']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

The next cell block is a copy and paste of the functions from the first part of this notebook. We have this cellblock here so that we can run code in this section without having to rerun the previous sections.

In [6]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    # Must shift by 14 days because we do not want to let there be temporal leakage in our evaluations
    df['ma7'] = df['y'].shift(14).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(14).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(14).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(14).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(14).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(14).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(14).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(14).rolling(window=365).mean()
    
    return df


## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")


def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Remove overlapping columns to avoid join errors
    overlap = wd.columns.intersection(df.columns)
    wd = wd.drop(columns=overlap)
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df


date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

def add_new_lags(df, x):
    # lags
    target_map = df[x].to_dict()
    df[f'{x}lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df[f'{x}lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df[f'{x}lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df[f'{x}lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df[f'{x}lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df[f'{x}lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df[f'{x}lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df[f'{x}lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    return df

In [7]:
ALL_FEATURES = {'dayofweek', 'quarter', 'month', 'year', 'dayofyear',
       'dayofmonth', 'weekofyear', 'dayofweek_sin', 'dayofweek_cos',
       'month_sin', 'month_cos', 'lag15', 'lag16', 'lag30', 'lag60', 'lag90',
       'lag120', 'lag150', 'lag180', 'lag362', 'lag363', 'lag364', 'lag365',
       'lag366', 'lag367', 'lag730', 'lag1095', 'lag1460', 'lag1825', 'ma7',
       'ma30', 'ma60', 'ma90', 'ma120', 'ma150', 'ma180', 'ma365',
       'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
       'apparent_temperature_max', 'apparent_temperature_min',
       'apparent_temperature_mean', 'precipitation_sum', 'snowfall_sum',
       'apparent_temperature_min_lag14', 'apparent_temperature_min_lag15',
       'apparent_temperature_min_lag16', 'apparent_temperature_min_lag17',
       'apparent_temperature_min_lag18', 'apparent_temperature_min_lag19',
       'apparent_temperature_min_lag20', 'apparent_temperature_min_lag21',
       'apparent_temperature_min_lag30', 'apparent_temperature_min_lag60',
       'apparent_temperature_min_lag90', 'apparent_temperature_min_lag120',
       'apparent_temperature_min_lag150', 'apparent_temperature_min_lag180',
       'apparent_temperature_min_lag210', 'apparent_temperature_min_lag240',
       'apparent_temperature_min_lag270', 'apparent_temperature_min_lag300',
       'apparent_temperature_min_lag330', 'apparent_temperature_min_lag360',
       'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730',
       'temperature_2m_max_lag14', 'temperature_2m_max_lag30',
       'temperature_2m_max_lag60', 'is_federal_holiday', 'Trash_Law',
       'New_Trash_Law', 'Rat_Mitigation_Zone', 'Rat_Czar_Appointed',
       'residuals', 'residualslag15', 'residualslag16', 'residualslag17',
       'residualslag18', 'residualslag19', 'residualslag20', 'residualslag21',
       'residualslag30', 'residualslag365', 'residualslag730', 'trend',
       'yhat_lower', 'yhat_upper'}

# ALL_FEATURES =['lag60', 'ma7', 'yhat_lower', 'temperature_2m_max_lag60', 'dayofweek', 
#                'apparent_temperature_min_lag360', 'dayofyear', 'apparent_temperature_min_lag120', 'Trash_Law', 'quarter', 'month_sin', 'ma60', 'Rat_Mitigation_Zone', 'temperature_2m_max_lag30', 'lag730', 'apparent_temperature_max', 'apparent_temperature_min', 
#                'residualslag365', 'lag366', 'lag363', 'lag364', 'apparent_temperature_min_lag180', 
#                'apparent_temperature_min_lag240', 'apparent_temperature_min_lag18', 'apparent_temperature_min_lag90', 'dayofweek_cos']

In [8]:
save = rs['ds'].copy().values
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)
rs['ds']=save
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')
rs

,y,ds,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,...,apparent_temperature_min_lag365,apparent_temperature_min_lag730,temperature_2m_max_lag14,temperature_2m_max_lag30,temperature_2m_max_lag60,is_federal_holiday,Trash_Law,New_Trash_Law,Rat_Mitigation_Zone,Rat_Czar_Appointed
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,4,2020-01-01,2,1,1,2020,1,1,1,0.974928,...,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0
2020-01-02,7,2020-01-02,3,1,1,2020,2,2,1,0.433884,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-03,16,2020-01-03,4,1,1,2020,3,3,1,-0.433884,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-04,10,2020-01-04,5,1,1,2020,4,4,1,-0.974928,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-05,5,2020-01-05,6,1,1,2020,5,5,1,-0.781831,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,19,2025-02-24,0,1,2,2025,55,24,9,0.000000,...,-11.1,-10.8,0.2,-2.2,-0.2,0,1,1,1,1
2025-02-25,20,2025-02-25,1,1,2,2025,56,25,9,0.781831,...,-6.9,-8.1,-0.4,4.1,2.5,0,1,1,1,1
2025-02-26,15,2025-02-26,2,1,2,2025,57,26,9,0.974928,...,-4.1,-7.1,0.6,2.0,7.5,0,1,1,1,1


In [9]:
import logging
logging.getLogger("cmdstanpy").disabled = True

In [10]:
import optuna

def objective(trial):
    print("Trial starting:", trial.number)

    # Prophet Hyperparameters
    prophet_params = {
        "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.1, 0.5, log=True),
        "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.1, 10, log=True),
        "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 10, log=True),
    }

    # XGBoost Hyperparameters
    xgb_params = {"device": "cuda",
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 1, log=True),
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10, log=True),
        "objective": "reg:squarederror",
        "random_state": 42
    }


    # Feature Selection
    selected_features = []

    for feature in ALL_FEATURES:
        if trial.suggest_categorical(feature, [True, False]):
            selected_features.append(feature)


    # prevent Optuna from accidentally picking no features
    if len(selected_features) == 0: 
        selected_features.append("trend")


    results = []

    for i, (train_index, test_index) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_index].copy()
        test = rs.iloc[test_index].copy()

        model = Prophet(**prophet_params, holidays=holidays)
        model.add_country_holidays(country_name='US')
        model.fit(train)

        train_future = model.make_future_dataframe(periods=0, freq='D')
        train_forecast = model.predict(train_future)

        train_residuals = train['y'].values - train_forecast['yhat'].values

        residuals_df = pd.DataFrame({'ds': train['ds'],'y': train_residuals})

        train['residuals'] = train_residuals

        add_new_lags(train, 'residuals')

        train['trend'] = train_forecast['trend'].values
        train['yhat_lower'] = train_forecast['yhat_lower'].values
        train['yhat_upper'] = train_forecast['yhat_upper'].values

        X_train_residuals = train[selected_features]
        y_train_residuals = residuals_df['y']

        # Train XGBoost
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_train_residuals, y_train_residuals)

        # Prepare Test Data
        test['residuals'] = np.nan # need to add this otherwise it won't run

        dummy = pd.concat([train, test], axis=0)

        add_new_lags(dummy, 'residuals') # need to add the lags that test can actually see

        test = dummy.iloc[test_index].copy() # cut out the test set again

        # Prophet Forecast
        future = model.make_future_dataframe(periods=len(test), freq='D')
        prophet_forecast = model.predict(future)

        # add outputs of Prophet for use in the XGBoost model
        test.loc[:, 'trend'] = prophet_forecast[-len(test):]['trend'].values
        test.loc[:, 'yhat_lower'] = prophet_forecast[-len(test):]['yhat_lower'].values
        test.loc[:, 'yhat_upper'] = prophet_forecast[-len(test):]['yhat_upper'].values

        X_test = test[selected_features]

        xgb_residual_preds = xgb_model.predict(X_test)

        y_pred = np.round(prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds)
        y_true = test['y'].values

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        results.append(rmse)

        # # Report intermediate result to Optuna
        # trial.report(rmse, step=i)

        # # Check if trial should be pruned
        # if trial.should_prune():
        #     raise TrialPruned()

    return np.mean(results)

In [ ]:
splits = 26 # from among 7, 13, 26, 


# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=splits, test_size=14)

study = optuna.create_study(
    direction="minimize",
    study_name="hybrid_model_feature_parameter_search",
    storage=f"sqlite:///xgbprophet_model{splits}.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=200, n_jobs=-1)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


best_features = [f for f in ALL_FEATURES if study.best_params.get(f, False)]

best_hyperparams = {k: v for k, v in study.best_params.items() if k not in ALL_FEATURES}
print("Selected Features:")
print(best_features)

print("\nBest Hyperparameters:")
print(best_hyperparams)

[I 2026-03-15 17:15:50,262] Using an existing study with name 'hybrid_model_feature_parameter_search' instead of creating a new one.


Trial starting: 302
Trial starting: 292
Trial starting: 289
Trial starting: 291
Trial starting: 294
Trial starting: 299
Trial starting: 297
Trial starting: 298
Trial starting: 293
Trial starting: 290
Trial starting: 296
Trial starting: 295
Trial starting: 287
Trial starting: 300
Trial starting: 301
Trial starting: 288


[I 2026-03-15 17:35:53,987] Trial 302 finished with value: 13.71340864823187 and parameters: {'changepoint_prior_scale': 0.4049508860436023, 'seasonality_prior_scale': 0.8126395637319624, 'holidays_prior_scale': 0.29783156885675377, 'n_estimators': 892, 'max_depth': 7, 'learning_rate': 0.0055887028678074544, 'subsample': 0.6689736500530491, 'colsample_bytree': 0.9541099048802497, 'gamma': 1.4121310519464036, 'min_child_weight': 10, 'reg_lambda': 0.13499303155895848, 'reg_alpha': 0.14608175983781668, 'month_sin': True, 'apparent_temperature_min_lag150': False, 'apparent_temperature_min_lag21': False, 'apparent_temperature_min_lag16': True, 'apparent_temperature_min_lag365': True, 'dayofweek_cos': False, 'apparent_temperature_min_lag60': True, 'residualslag19': False, 'dayofweek_sin': True, 'Rat_Czar_Appointed': False, 'apparent_temperature_min_lag730': True, 'yhat_upper': False, 'apparent_temperature_min_lag360': False, 'apparent_temperature_min_lag18': False, 'temperature_2m_mean': Fal

Trial starting: 303


[I 2026-03-15 17:36:18,009] Trial 298 finished with value: 13.748086857289024 and parameters: {'changepoint_prior_scale': 0.39713603900429384, 'seasonality_prior_scale': 0.8353949742279996, 'holidays_prior_scale': 0.31160500787086914, 'n_estimators': 894, 'max_depth': 7, 'learning_rate': 0.005295844355245781, 'subsample': 0.6930927833834213, 'colsample_bytree': 0.9485814554079968, 'gamma': 1.350038480457613, 'min_child_weight': 10, 'reg_lambda': 0.13656116105705207, 'reg_alpha': 0.15453532527475738, 'month_sin': True, 'apparent_temperature_min_lag150': False, 'apparent_temperature_min_lag21': False, 'apparent_temperature_min_lag16': True, 'apparent_temperature_min_lag365': True, 'dayofweek_cos': False, 'apparent_temperature_min_lag60': True, 'residualslag19': False, 'dayofweek_sin': True, 'Rat_Czar_Appointed': False, 'apparent_temperature_min_lag730': True, 'yhat_upper': False, 'apparent_temperature_min_lag360': False, 'apparent_temperature_min_lag18': False, 'temperature_2m_mean': Fal

...

Trial starting: 346


[I 2026-03-15 18:25:44,661] Trial 330 finished with value: 4.346503098608749 and parameters: {'changepoint_prior_scale': 0.38529257218528495, 'seasonality_prior_scale': 1.0255285499743243, 'holidays_prior_scale': 0.15599694096576397, 'n_estimators': 834, 'max_depth': 7, 'learning_rate': 0.0037988333857763195, 'subsample': 0.6753192439124094, 'colsample_bytree': 0.8327033764644776, 'gamma': 1.1745603263771471, 'min_child_weight': 9, 'reg_lambda': 0.10020776386394754, 'reg_alpha': 0.4269043717225044, 'month_sin': True, 'apparent_temperature_min_lag150': False, 'apparent_temperature_min_lag21': False, 'apparent_temperature_min_lag16': False, 'apparent_temperature_min_lag365': True, 'dayofweek_cos': False, 'apparent_temperature_min_lag60': True, 'residualslag19': False, 'dayofweek_sin': True, 'Rat_Czar_Appointed': False, 'apparent_temperature_min_lag730': True, 'yhat_upper': False, 'apparent_temperature_min_lag360': False, 'apparent_temperature_min_lag18': False, 'temperature_2m_mean': Fal

Trial starting: 347


[I 2026-03-15 18:27:15,373] Trial 332 finished with value: 4.380251583476573 and parameters: {'changepoint_prior_scale': 0.4655640222101024, 'seasonality_prior_scale': 1.032742249997796, 'holidays_prior_scale': 0.15721817079423808, 'n_estimators': 826, 'max_depth': 7, 'learning_rate': 0.0025304559984154224, 'subsample': 0.7351267478108302, 'colsample_bytree': 0.8320883711171528, 'gamma': 3.9651360147536336, 'min_child_weight': 9, 'reg_lambda': 0.12273857966792866, 'reg_alpha': 0.30619894454117014, 'month_sin': True, 'apparent_temperature_min_lag150': False, 'apparent_temperature_min_lag21': False, 'apparent_temperature_min_lag16': False, 'apparent_temperature_min_lag365': True, 'dayofweek_cos': False, 'apparent_temperature_min_lag60': True, 'residualslag19': False, 'dayofweek_sin': True, 'Rat_Czar_Appointed': False, 'apparent_temperature_min_lag730': True, 'yhat_upper': False, 'apparent_temperature_min_lag360': False, 'apparent_temperature_min_lag18': False, 'temperature_2m_mean': Fals

Trial starting: 348


In [ ]:
best_features

In [ ]:
best_hyperparams

In [ ]:
prophet_keys = ["changepoint_prior_scale", "seasonality_prior_scale", "holidays_prior_scale"]

# extract subset
best_prophet_params = {k: best_hyperparams[k] for k in prophet_keys if k in best_hyperparams}


xgb_keys = ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'gamma', 'min_child_weight', 'reg_lambda', 'reg_alpha'] 

# extract subset
best_xgb_params = {k: best_hyperparams[k] for k in xgb_keys if k in best_hyperparams}




# Cross Validate if Hybrid Model Performs as well as Claimed


We cross validate to check that there is a genuine improvement on the model's performance. Again, we saw that Prophet performed the best and we want to compare Prophet vs. this Hybrid model vs. NeuralProphet. Since our work in 1modeling_experiments.ipynb did not even try to tune XGBoost's parameters and features to see if there is an improvement, we have done it here. 

In [ ]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs[rs['borough']=='MANHATTAN']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [ ]:
save = rs['ds'].copy().values
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)
rs['ds']=save
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')
rs

In [ ]:
FEATURES = best_features

In [ ]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()

    model = Prophet(**best_prophet_params, holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)

    train_future = model.make_future_dataframe(periods=0, freq='D')
    train_forecast = model.predict(train_future)

    train_residuals = train['y'].values - train_forecast['yhat'].values

    residuals_df = pd.DataFrame({'ds': train['ds'],'y': train_residuals})

    train['residuals'] = train_residuals

    add_new_lags(train, 'residuals')

    train['trend'] = train_forecast['trend'].values
    train['yhat_lower'] = train_forecast['yhat_lower'].values
    train['yhat_upper'] = train_forecast['yhat_upper'].values

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']

    # Train XGBoost
    xgb_model = xgb.XGBRegressor(**best_xgb_params)
    xgb_model.fit(X_train_residuals, y_train_residuals)

    # Prepare Test Data
    test['residuals'] = np.nan # need to add this otherwise it won't run

    dummy = pd.concat([train, test], axis=0)

    add_new_lags(dummy, 'residuals') # need to add the lags that test can actually see

    test = dummy.iloc[test_index].copy() # cut out the test set again

    # Prophet Forecast
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)

    # add outputs of Prophet for use in the XGBoost model
    test.loc[:, 'trend'] = prophet_forecast[-len(test):]['trend'].values
    test.loc[:, 'yhat_lower'] = prophet_forecast[-len(test):]['yhat_lower'].values
    test.loc[:, 'yhat_upper'] = prophet_forecast[-len(test):]['yhat_upper'].values

    X_test = test[FEATURES]

    xgb_residual_preds = xgb_model.predict(X_test)

    y_pred = np.round(prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds)
    y_true = test['y'].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    results.append(rmse)    
    # Store the results for this fold

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results, columns=["RMSE"])

In [ ]:
mean_rmse = prophet_xgb_results_df['RMSE'].mean()
prophet_xgb_results_df.loc['mean'] = [mean_rmse]

# Prophet + XGBoost Results

In [ ]:
prophet_xgb_results_df